# Lab: Build a TOR-Like Server Node

In this lab, you will build server nodes that work together to create a TOR-like network. Each node will listen on a specific port, decrypt incoming packets, and forward them to the next node or send the actual request if it is the last node. The nodes will ensure secure and anonymous communication by encrypting and decrypting the data at each step.

### Objectives:

1. **Listening on a Specific Port**: Each node will listen on a designated port for incoming connections.

2. **Receiving and Decrypting Packets**: When a node receives a connection, it will receive the packet and decrypt its layer of encryption.

3. **Forwarding to the Next Node**: 
    - If the decrypted packet contains an IP address and port, the node will forward the remaining encrypted packet to the next node in the circuit.
    - The packet forwarded will still be encrypted (it will be the second layer of encryption).

4. **Sending the Actual Request**:
    - If the node is the last in the circuit, upon decryption, it will reveal the actual HTTP request.
    - The node will send the HTTP request to the target server and obtain the response.

5. **Returning the Response**:
    - The node will return the response to the parent node, encrypting it with the key to maintain the security and anonymity of the communication. The response must follow the circuit until it gets to the client.

### Steps:

1. **Listening on a Specific Port**:
    - Set up each node to listen on a designated port for incoming connections.

2. **Receiving and Decrypting Packets**:
    - When a node receives a packet, it will decrypt its layer using its private key.

3. **Forwarding to the Next Node**:
    - If the decrypted packet contains an IP address and port, the node will forward the remaining encrypted packet to the next node in the circuit.
    - Ensure the packet remains encrypted for the next node.

4. **Sending the Actual Request**:
    - If the node is the last in the circuit, it will decrypt the packet to reveal the HTTP request.
    - Send the HTTP request to the target server and obtain the response.

5. **Returning the Response**:
    - Encrypt the response.
    - Send the encrypted response back through the circuit to the client.

### Tips:

Watchout with the lenght of the packets. Most encryption errors could be due this, so you'll maybe have to send and handle chunks. Every time the packet is encrypted, it's size will change

In [ ]:
# TOR-like Node Implementation

import socket
import threading
import ssl
from time import sleep
from cryptography.fernet import Fernet


def encrypt_message(message, session_key):
    f = Fernet(session_key)
    if isinstance(message, str):
        message = message.encode()
    return f.encrypt(message)


def decrypt_message(encrypted_message, session_key):
    return Fernet(session_key).decrypt(encrypted_message)


class Node:
    PORT_START = 5000
    
    def __init__(self, id, next_node=None, session_key=None):
        self.id = id
        self.port = self.PORT_START + id
        self.next_node = next_node
        self.session_key = session_key
        self.running = True

    def handle_client(self, conn, addr):
        try:
            # Receive all encrypted data
            conn.settimeout(5.0)
            data = b""
            while True:
                try:
                    chunk = conn.recv(4096)
                    if not chunk:
                        break
                    data += chunk
                except socket.timeout:
                    break
            
            if not data:
                return
            
            # Decrypt this layer
            decrypted = decrypt_message(data, self.session_key)
            
            if self.next_node:
                # Forward to next node and get response
                response = self.send_data(decrypted)
                # Re-encrypt and send back
                conn.sendall(encrypt_message(response, self.session_key))
            else:
                # Exit node - make actual HTTPS request
                context = ssl.create_default_context()
                host = self.extract_host(decrypted)
                
                with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as dest_socket:
                    with context.wrap_socket(dest_socket, server_hostname=host) as ssock:
                        ssock.connect((host, 443))
                        ssock.sendall(decrypted)
                        
                        response = b""
                        while True:
                            chunk = ssock.recv(4096)
                            if not chunk:
                                break
                            response += chunk
                
                # Encrypt and send back
                conn.sendall(encrypt_message(response, self.session_key))
                
        except Exception as e:
            print(f"Node {self.id} error: {e}")
        finally:
            conn.close()

    def extract_host(self, request_bytes):
        request_str = request_bytes.decode('utf-8')
        lines = request_str.split('\r\n')
        for line in lines:
            if line.startswith('Host: '):
                host = line.split(' ')[1]
                return host
        return None

    def send_data(self, data, s=None):
        if s is None:
            # Create new socket to next node
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
                sock.connect(('127.0.0.1', self.next_node.port))
                sock.sendall(data)
                sock.settimeout(10.0)
                
                response = b""
                while True:
                    try:
                        chunk = sock.recv(4096)
                        if not chunk:
                            break
                        response += chunk
                    except socket.timeout:
                        break
                return response
        else:
            # Send using existing socket
            s.sendall(data)
            return None

    def start(self):
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind(('127.0.0.1', self.port))
        s.listen(5)
        print(f"Node {self.id} listening on port {self.port}")
        
        while self.running:
            try:
                conn, addr = s.accept()
                threading.Thread(target=self.handle_client, args=(conn, addr), daemon=True).start()
            except OSError:
                break
        s.close()
    
    def stop(self):
        self.running = False

In [ ]:
# Start all nodes in background threads

def run_node(node):
    node.start()

# Start each node in a background thread
thread0 = threading.Thread(target=run_node, args=(node0,), daemon=True)
thread1 = threading.Thread(target=run_node, args=(node1,), daemon=True)
thread2 = threading.Thread(target=run_node, args=(node2,), daemon=True)

thread0.start()
thread1.start()
thread2.start()

print("All nodes started!")
print(f"- Node 0 listening on port {node0.port} (exit node)")
print(f"- Node 1 listening on port {node1.port} (forwards to {node0.port})")
print(f"- Node 2 listening on port {node2.port} (forwards to {node1.port})")
print("\nReady to receive packets.")

In [ ]:
# Test client - Send HTTP request through the TOR-like circuit

import socket

http_request = b"""GET / HTTP/1.1\r
Host: example.com\r
Connection: close\r
\r
"""

# Onion encryption: Encrypt in REVERSE order of the path
# Path: client -> node2 -> node1 -> node0 -> destination
# Decrypt order: node2 decrypts first, then node1, then node0
# So we encrypt: key1 (node0) first, then key2 (node1), then key3 (node2)
layer1 = encrypt_message(http_request, key1)   # Inner layer (node0 decrypts)
layer2 = encrypt_message(layer1, key2)         # Middle layer (node1 decrypts)
onion_packet = encrypt_message(layer2, key3)   # Outer layer (node2 decrypts)

print(f"Original: {len(http_request)} bytes")
print(f"Encrypted: {len(onion_packet)} bytes")

# Send through circuit to node2 (port 5002)
try:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.connect(('127.0.0.1', 5002))
        sock.sendall(onion_packet)
        sock.settimeout(10.0)
        
        # Receive encrypted response (needs to be decrypted in reverse order)
        response = sock.recv(8192)
        print(f"Received {len(response)} encrypted bytes")
        
        # Decrypt response layers (reverse order: key3, key2, key1)
        layer3_resp = decrypt_message(response, key3)
        layer2_resp = decrypt_message(layer3_resp, key2)
        final_response = decrypt_message(layer2_resp, key1)
        
        print(f"Final response: {len(final_response)} bytes")
        print("\n--- Response Preview ---")
        print(final_response[:500].decode('utf-8', errors='ignore'))
        
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Create 3-node circuit: client -> node2 -> node1 -> node0 (exit) -> destination

# Generate unique session keys for each node
key1 = Fernet.generate_key()  # For node0 (exit node)
key2 = Fernet.generate_key()  # For node1
key3 = Fernet.generate_key()  # For node2 (entry node)

# Create nodes in a chain
# Node 0 is the EXIT node (makes actual HTTPS request)
node0 = Node(id=0, session_key=key1)
# Node 1 forwards to node0
node1 = Node(id=1, next_node=node0, session_key=key2)
# Node 2 forwards to node1
node2 = Node(id=2, next_node=node1, session_key=key3)

print("Nodes created!")
print(f"Node 0: port {node0.port} (exit node)")
print(f"Node 1: port {node1.port} -> forwards to {node0.port}")
print(f"Node 2: port {node2.port} -> forwards to {node1.port}")